---
title: "Exercise 4"
editor: source
editor_options: 
  chunk_output_type: console
jupyter: ir
---

# Overlapping all the datasets

## Learning Objectives

By the end of this exercise, you will be able to:
- Load and explore genomic data using Bioconductor classes such as `GRangesList` and `RangedSummarizedExperiment`.
- Subset and filter genomic features based on annotation or genomic coordinates.
- Perform basic overlap and distance-based queries between genomic intervals.
- Integrate multi-assay data (e.g., chromatin marks and gene expression) using shared genomic coordinates.


## Load Libraries

In [ ]:
knitr::opts_chunk$set(warning = FALSE) 

In [ ]:
#| warning: false
library(ComplexHeatmap)
library(EnrichedHeatmap)
library(SummarizedExperiment)
library(gUtils)
library(rtracklayer)
library(data.table)
library(parallel)
library(R.utils)
library(circlize)
library(dplyr)
library(tibble)

## SE objects

In [ ]:
atac <- readRDS("data/atac_se.rds")
rna <- readRDS("data/rna_se.rds")
h3k4me3 <- readRDS("data/h3k4me3_se.rds")
h3k4me1 <- readRDS("data/h3k4me1_se.rds")
h3k27me3 <- readRDS("data/h3k27me3_se.rds")
h3k27ac <- readRDS("data/h3k27ac_se.rds")

## Overlap matrix
To make the `overlapMatrix` (you could also say `overlapRanges`), we need to identify a target. 

## Function to merge Genomic Ranges

In [ ]:
# Merge metadata from two GRanges objects based on overlaps
metaGR <- function(gr1, gr2, minOverlap = 1) {
  # Initialize metadata with gr1's metadata
  mcols_out <- mcols(gr1)

  # Find overlaps (keep all from gr1)
  hits <- findOverlaps(gr1, gr2, minoverlap = minOverlap)
  idx1 <- subjectHits(hits)
  idx2 <- queryHits(hits)

  # Prepare new metadata to be added
  new_mcols <- DataFrame(matrix(NA, nrow = length(gr1), ncol = ncol(mcols(gr2))))
  colnames(new_mcols) <- colnames(mcols(gr2))
  
  # Fill metadata only for overlaps
  new_mcols[idx2, ] <- mcols(gr2)[idx1, , drop = FALSE]

  # Combine original and new metadata
  mcols(gr1) <- cbind(mcols_out, new_mcols)
  
  return(gr1)
}

## ATAC-Seq

In [ ]:
# Extract rowRanges and filter peaks with strong signal
rd_atac <- rowRanges(atac)
rd_atac <- rd_atac[abs(rd_atac$logFC) >= 0.5 & rd_atac$qvalue <= 0.1]
colnames(elementMetadata(rd_atac)) <- paste("ATAC", colnames(elementMetadata(rd_atac)), sep = "_")

## RNA

In [ ]:
# Filter differentially expressed genes
rd_rna <- rowRanges(rna)
rd_rna <- rd_rna[abs(rd_rna$logFC) >= 0.5 & rd_rna$qvalue <= 0.1]
colnames(elementMetadata(rd_rna)) <- paste("RNA", colnames(elementMetadata(rd_rna)), sep = "_")
overlap <- metaGR(gr1 = rd_atac, gr2 = rd_rna, minOverlap = 10)

## ChIP-Seq
### H3K4me3

In [ ]:
rd_h3k4me3 <- rowRanges(h3k4me3)
rd_h3k4me3 <- rd_h3k4me3[abs(rd_h3k4me3$logFC) >= 0.5 & rd_h3k4me3$qvalue <= 0.1]
colnames(elementMetadata(rd_h3k4me3)) <- paste("H3K4me3", colnames(elementMetadata(rd_h3k4me3)), sep = "_")
overlap <- metaGR(gr1 = overlap, gr2 = rd_h3k4me3, minOverlap = 10)

### H3K4me1

In [ ]:
rd_h3k4me1 <- rowRanges(h3k4me1)
rd_h3k4me1 <- rd_h3k4me1[abs(rd_h3k4me1$logFC) >= 0.5 & rd_h3k4me1$qvalue <= 0.1]
colnames(elementMetadata(rd_h3k4me1)) <- paste("H3K4me1", colnames(elementMetadata(rd_h3k4me1)), sep = "_")
overlap <- metaGR(gr1 = overlap, gr2 = rd_h3k4me1, minOverlap = 10)

### H3K27me3

In [ ]:
rd_h3k27me3 <- rowRanges(h3k27me3)
rd_h3k27me3 <- rd_h3k27me3[abs(rd_h3k27me3$logFC) >= 0.5 & rd_h3k27me3$qvalue <= 0.1]
colnames(elementMetadata(rd_h3k27me3)) <- paste("H3K27me3", colnames(elementMetadata(rd_h3k27me3)), sep = "_")
overlap <- metaGR(gr1 = overlap, gr2 = rd_h3k27me3, minOverlap = 10)

### H3K27ac

In [ ]:
rd_h3k27ac <- rowRanges(h3k27ac)
rd_h3k27ac <- rd_h3k27ac[abs(rd_h3k27ac$logFC) >= 0.5 & rd_h3k27ac$qvalue <= 0.1]
colnames(elementMetadata(rd_h3k27ac)) <- paste("H3K27ac", colnames(elementMetadata(rd_h3k27ac)), sep = "_")
overlap <- metaGR(gr1 = overlap, gr2 = rd_h3k27ac, minOverlap = 10)

## Save data

In [ ]:
# Add a name field and save the final integrated object
overlap$name <- paste("peak", 1:length(overlap), sep = "_")
names(overlap) <- overlap$name

dir.create("output")
saveRDS(object = overlap, file = "output/overlap_data.rds")

### Snapshot of overlaMatrix

In [ ]:
overlap[1:3]

# Normalized metrics for plotting

Recommendations based on ENCODE Project Consortium and other literature:

| Assays                     | Typical Window Size (± kb) | Reason / Notes                                 |
| -------------------------- | -------------------------- | ---------------------------------------------- |
| **ATAC-seq**               | ±1 kb                      | Narrow peaks, accessibility centered on summit |
| **H3K4me3**                | ±1 kb to ±2 kb             | Promoter mark, sharp and narrow peaks          |
| **H3K4me1**                | ±2 kb to ±5 kb             | Enhancer-associated, moderately broad          |
| **H3K27me3**               | ±5 kb to ±10 kb            | Broad repressive domains                       |
| **H3K27ac**                | ±2 kb to ±5 kb             | Active enhancers/promoters, broader peaks      |

We can check the `mean` and `median` of each histone marks and make the `normalizedMatrix` accordingly.

In [ ]:
# Calculate mean and median widths for each dataset

summary_table <- tibble(
  Assay = c("ATAC-seq", "H3K4me3", "H3K4me1", "H3K27me3", "H3K27ac"),
  Mean_Width = c(
    mean(width(rd_atac)),
    mean(width(rd_h3k4me3)),
    mean(width(rd_h3k4me1)),
    mean(width(rd_h3k27me3)),
    mean(width(rd_h3k27ac))
  ),
  Median_Width = c(
    median(width(rd_atac)),
    median(width(rd_h3k4me3)),
    median(width(rd_h3k4me1)),
    median(width(rd_h3k27me3)),
    median(width(rd_h3k27ac))
  ),
  Min_Width = c(
    min(width(rd_atac)),
    min(width(rd_h3k4me3)),
    min(width(rd_h3k4me1)),
    min(width(rd_h3k27me3)),
    min(width(rd_h3k27ac))
  ),
  Max_Width = c(
    max(width(rd_atac)),
    max(width(rd_h3k4me3)),
    max(width(rd_h3k4me1)),
    max(width(rd_h3k27me3)),
    max(width(rd_h3k27ac))
  )
)

print(summary_table)

As we restricted the data for this course to just 2 chromosomes, maximum peaks are around 1kb wide. Hence, we will go ahead with `+/- 1kb` windows for our analysis.


## Taking 1000 bp around the mid of ATAC-peaks

In [ ]:
# Extract midpoints of peaks to create windows for visualization
mid_peaks <- gr.mid(overlap)

## ATAC

In [ ]:
# Load ATAC bigWig files
atac_files <- list.files("data", pattern = "ATAC", full.names = TRUE)
names(atac_files) <- gsub(pattern = "\\.bw", replacement = "", x = basename(atac_files))
atac_bw <- lapply(atac_files, function(x){
  a <- rtracklayer::import(x)
  a <- a[seqnames(a) %in% c("chr1", "chr2")]
  a
})

# Normalize signal around mid_peaks
mat_AS <- lapply(atac_bw, FUN = function(x) {
  normalizeToMatrix(x, mid_peaks,
    extend = 1000,
    value_column = "score",
    include_target = TRUE,
    mean_mode = "w0",
    w = 20, 
    smooth = T,
    background = 0
  )
})

saveRDS(mat_AS, file = "output/mat_atac.rds")

## ChIP

### H3K4me3

In [ ]:
h3k4me3_files <- list.files("data", pattern = "H3K4me3", full.names = TRUE)
names(h3k4me3_files) <- gsub(pattern = "\\.bw", replacement = "", x = basename(h3k4me3_files))
h3k4me3_bw <- lapply(h3k4me3_files, function(x){
  a <- rtracklayer::import(x)
  a <- a[seqnames(a) %in% c("chr1", "chr2")]
  a
})

mat_h3k4me3 <- lapply(h3k4me3_bw, FUN = function(x) {
  normalizeToMatrix(x, mid_peaks,
    extend = 1000,
    value_column = "score",
    include_target = TRUE,
    mean_mode = "w0",
    w = 20,
    smooth = T,
    background = 0
  )
})

saveRDS(mat_h3k4me3, file = "output/mat_h3k4me3.rds")

### H3K4me1

In [ ]:
h3k4me1_files <- list.files("data", pattern = "H3K4me1", full.names = TRUE)
names(h3k4me1_files) <- gsub(pattern = "\\.bw", replacement = "", x = basename(h3k4me1_files))
h3k4me1_bw <- lapply(h3k4me1_files, function(x){
  a <- rtracklayer::import(x)
  a <- a[seqnames(a) %in% c("chr1", "chr2")]
  a
})

mat_h3k4me1 <- lapply(h3k4me1_bw, FUN = function(x) {
  normalizeToMatrix(x, mid_peaks,
    extend = 1000,
    value_column = "score",
    include_target = TRUE,
    mean_mode = "w0",
    w = 20,
    smooth = T,
    background = 0
  )
})

saveRDS(mat_h3k4me1, file = "output/mat_h3k4me1.rds")

### H27K4me3

In [ ]:
h3k27me3_files <- list.files("data", pattern = "H3K27me3", full.names = TRUE)
names(h3k27me3_files) <- gsub(pattern = "\\.bw", replacement = "", x = basename(h3k27me3_files))
h3k27me3_bw <- lapply(h3k27me3_files, function(x){
  a <- rtracklayer::import(x)
  a <- a[seqnames(a) %in% c("chr1", "chr2")]
  a
})

mat_h3k27me3 <- lapply(h3k27me3_bw, FUN = function(x) {
  normalizeToMatrix(x, mid_peaks,
    extend = 1000,
    value_column = "score",
    include_target = TRUE,
    mean_mode = "w0",
    w = 20,
    smooth = T,
    background = 0
  )
})

saveRDS(mat_h3k27me3, file = "output/mat_h3k27me3.rds")

### H27K4ac

In [ ]:
h3k27ac_files <- list.files("data", pattern = "H3K27ac", full.names = TRUE)
names(h3k27ac_files) <- gsub(pattern = "\\.bw", replacement = "", x = basename(h3k27ac_files))
h3k27ac_bw <- lapply(h3k27ac_files, function(x){
  a <- rtracklayer::import(x)
  a <- a[seqnames(a) %in% c("chr1", "chr2")]
  a
})

mat_h3k27ac <- lapply(h3k27ac_bw, FUN = function(x) {
  normalizeToMatrix(x, mid_peaks,
    extend = 1000,
    value_column = "score",
    include_target = TRUE,
    mean_mode = "w0",
    w = 20,
    smooth = T,
    background = 0
  )
})

saveRDS(mat_h3k27ac, file = "output/mat_h3k27ac.rds")

## RNA

In [ ]:
# Match overlapping peak names with RNA-seq gene names
tmp <- elementMetadata(overlap)[,c("RNA_Row.names", "name")]

# Get logCPM matrix
counts <- assay(rna, "logCPM")
counts <- assay(rna, "logCPM") - rowMeans(
  assay(rna, "logCPM")[, grep(pattern = "11half", x = colnames(assay(rna, "logCPM")),
                              value = T)]
)

# Merge gene expression values with peaks
mat_RNA <- merge(tmp, counts, by.x = "RNA_Row.names", by.y = "row.names", all.x = T)

# Set rownames to peak names and cleanup
rownames(mat_RNA) <- mat_RNA$name
mat_RNA <- mat_RNA[,-c(1:2)]
colnames(mat_RNA) <- gsub(pattern = ".tsv.gz", replacement = "", x = colnames(mat_RNA))
mat_RNA <- data.matrix(mat_RNA)
mat_RNA <- mat_RNA[names(overlap),]

# Save data
saveRDS(mat_RNA, file = "output/mat_rna.rds")

## WGBS

In [ ]:
# Bisulfite seq coverage files after methylation call
bs_files <- list.files("data", pattern = "WGBS", full.names = TRUE)
names(bs_files) <- gsub(pattern = "\\.bed.gz", replacement = "", x = basename(bs_files))

# Reading coverage
bs_cov <- lapply(bs_files, function(x) {
  GRanges(  
  fread(
      input = x,
      sep = " ", quote = F, stringsAsFactors = F,
      data.table = FALSE, nThread = parallel::detectCores(), showProgress = F,
      col.names = c("seqnames", "start", "end", "cov", "Me", "Un", "meth")
    )
  )
})

# Normalized matrics
mat_bs <- lapply(bs_cov, FUN = function(x) {
    normalizeToMatrix(x, mid_peaks,
      extend = 1000,
      value_column = "meth",
      include_target = TRUE,
      smooth = TRUE,
      mean_mode = "absolute",
      background = NA
    )
})

saveRDS(mat_bs, file = "output/mat_bs.rds")

## Question 1
**Make an `EnrichedHeatmap`** of methylation data.

:::{.callout-tip collapse="true"}

In [ ]:
EnrichedHeatmap(
  mat = mat_bs$WGBS_11half,    # normalized matrix
  name = "E11.5",              # Name for the plot
  width = unit(4, "cm"),       # Width of the heatmap
  height = unit(8, "cm")      # Height of the heatmap
) + EnrichedHeatmap(
  mat = mat_bs$WGBS_15half,    # normalized matrix
  name = "E15.5",              # Name for the plot
  width = unit(4, "cm"),       # Width of the heatmap
  height = unit(8, "cm")      # Height of the heatmap
)

:::

:::{.callout-important}
We have not performed differential analysis for DNA methylation data. If you want, you can. Here, we are using DNA methylation as an observartory mark.
:::